In [46]:
import os

from dotenv import load_dotenv

from anthropic import Anthropic

load_dotenv()


client = Anthropic(api_key=os.getenv("ANTHROPIC_AUTH_TOKEN"), base_url=os.getenv("ANTHROPIC_BASE_URL"))

model = os.getenv("MODEL_NAME")


In [42]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, content):
    # `content` is the FULL block list from response.content (thinking + text),
    # not just the text. Thinking blocks must be passed back unchanged in the
    # conversation history: their signature is validated by the API, and the
    # model relies on its own earlier reasoning in follow-up turns.
    assistant_message = {"role": "assistant", "content": content}
    messages.append(assistant_message)

def get_text(response):
    # Filter blocks by type instead of hardcoding an index: the response
    # usually contains a thinking block before the text one, and their
    # number/order is not guaranteed.
    return next(b.text for b in response.content if b.type == "text")

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system
    
    # Return the full response object: the caller shows get_text(response)
    # to the user, but stores response.content in the history.
    return client.messages.create(**params)

In [43]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
response = chat(messages)
answer = get_text(response)  # text for display

# Add Claude's response to the conversation history.
# Full content (thinking + text), so thinking blocks stay in the context.
add_assistant_message(messages, response.content)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_response = chat(messages)
final_answer = get_text(final_response)

final_answer

'Quantum computing is an approach to information processing that exploits the ability of quantum bits to exist in multiple states at once, allowing parallel exploration of vast solution spaces that would be impossible for traditional binary systems.'

In [45]:
final_answer

'Quantum computing is an approach to information processing that exploits the ability of quantum bits to exist in multiple states at once, allowing parallel exploration of vast solution spaces that would be impossible for traditional binary systems.'